In [32]:
import pandas as pd
import re
from collections import Counter
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from google.colab import files

## Análisis de frecuencias

In [33]:
nltk.download('stopwords')


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [34]:
df_texto = pd.read_excel("Pertinencia-Base.xlsx")

In [35]:

columnas_utiles = [
    "Código de inscripción",
    "Modalidades",
    "Nombre",
    "Descriptores",
    "Objetivos específicos",
    "Descripción",
    "Objetivo general",
    "Población",
    "Subtemáticas",
    "Temáticas"
]

df_texto = df_texto[columnas_utiles]


In [36]:
df_pertinencias = pd.read_excel("Resultado_2025_pertinencia_entre_temas_proyectos_y_prioridades_ucr_y_nacionales.xlsx")
print(df_pertinencias["I.02"].nunique())

1201


In [37]:
df_pertinencias = df_pertinencias.drop_duplicates(
    subset=["I.02"]
)

df_final = df_texto.merge(
    df_pertinencias,
    left_on="Código de inscripción",
    right_on="I.02"
)

In [38]:
df_final

,Código de inscripción,Modalidades,Nombre,Descriptores,Objetivos específicos,Descripción,Objetivo general,Población,Subtemáticas,Temáticas,...,F.7.2 Descarbonización,F.7.3 Contribuye a la seguridad alimentaria,"G.1.1 Vinculación académico, investigaci on y la sociedad",G.2.1 Desarrollo territorial integral,G.2.2 Fortalecimiento de Sedes que desarrolle las comunidades,G.3.1 Programas y proyectos para sociedad inclusiva,H.5.1 Atención cantones del país debajo índice de desarrollo social (IDS),H.5.2 Mejora rendimiento académico estudiantes de cantones de bajo IDS,H.6.1 Incidencia en la política pública y la transformación social para el desarrollo humano sostenible con coordinación local,H.6.3 Fortalecimiento interuniversitario de emprendimientos regionales
0,EC-4,Cultura y Patrimonio,"Cátedra Amighetti: arte, política y cultura po...",Arte popular|Artes|Artes plásticas|Artes visua...,"1: Fomentar el estudio de saberes informales, ...","Proyecto de extensión cultural y educativo, ge...",Objetivo general:\nDesarrollar espacios de enc...,"Comunidad artística, sector cultura, público e...",Fortalecimiento educativo : Materiales didácti...,Artes|Derechos humanos|Desarrollo comunitario ...,...,0,0,0,0,0,0,0,0,0,0
1,EC-5,Cultura y Patrimonio,Producciones interdisciplinarias,Artes visuales|Enfoque interdisciplinario|Inte...,1: Promover el trabajo interdisciplinario entr...,El proyecto Producciones Interdisciplinarias e...,Desarrollar un espacio dentro de la Facultad d...,"Público en general, estudiantes de las carrera...",Artes : Artes escénicas|Artes : Artes plástica...,Artes,...,0,0,0,0,0,0,0,0,0,0
2,EC-11,Cultura y Patrimonio,Museo Regional Omar Salazar Obando de Turrialba,Sin descriptores asociados.,1: Mostrar a las comunidades de la región el p...,El Museo Regional Omar Salazar Obando de Turri...,"Promover la recuperación, protección y conserv...",Sin población registrada.,Fortalecimiento educativo : Materiales didácti...,Fortalecimiento educativo|Tradiciones,...,0,0,0,0,0,0,0,1,0,0
3,EC-16,Cultura y Patrimonio,Banda de la Sede de Occidente,Actividad cultural|Arte nacional|Cultura|Cultu...,1: Propiciar la participación de estudiantes u...,Se refiere a una orquesta de vientos y percusi...,Contribuir mediante la música sinfónica con el...,"La población beneficiada es, principalmente, ...",Artes : Audiovisuales,Artes,...,0,0,0,0,0,0,0,0,0,0
4,EC-17,Cultura y Patrimonio,"SET (Sala de Exposiciones Temporales, Museo Re...",Acceso a la educación|Acción cultural|Activida...,1: Facilitar espacios expositivos para artista...,La Sala de Exposiciones Temporales es un espac...,Contribuir al enriquecimiento cultural de la R...,"En lo que a población beneficiaria se refiere,...",Artes : Artes plásticas y diseño,Artes,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
573,TC-785,Trabajo Comunal Universitario,Pedaleando por los Derechos: Fortaleciendo la ...,Adolescencia|Cuidado del niño|Derecho a la sal...,1: Identificar las necesidades y los desafíos ...,El proyecto de Trabajo Comunal Universitario q...,"Promover los derechos de las niñas, niños y pe...","La población beneficiaria serán niñas, niños, ...",Derechos humanos : Democracia participativa|De...,Derechos humanos,...,0,0,0,0,0,0,0,0,0,0
574,TC-786,Trabajo Comunal Universitario,Sexualidades y Derechos Humanos para personas ...,Abuso sexual|Acoso|Comportamiento sexual|Derec...,1: Construir procesos permanentes de mapeo de ...,"El proyecto TCU "" Sexualidades y Derechos Huma...","Fomentar procesos de promoción, educación y se...",- Población que vive la niñez y la adolescenci...,Derechos humanos : Diversidad sexual,Derechos humanos,...,0,0,0,0,0,0,0,0,0,0
575,TC-787,Trabajo Comunal Universitario,Salud igualitaria: desafiando la brecha de gén...,Aporte educacional|Salud de la mujer,1: Sensibilizar sobre sus procesos fisiológico...,La anemia es un problema de salud pública sign...,Contribuir con la salud y el bienestar general...,Pob

In [39]:
df_final["texto"] = (
    df_final["Modalidades"].fillna('') + " " +
    df_final["Nombre"].fillna('') + " " +
    df_final["Descriptores"].fillna('') + " " +
    df_final["Objetivos específicos"].fillna('') + " " +
    df_final["Descripción"].fillna('') + " " +
    df_final["Objetivo general"].fillna('') + " " +
    df_final["Población"].fillna('') + " " +
    df_final["Subtemáticas"].fillna('') + " " +
    df_final["Temáticas"].fillna('')
)

In [40]:
# Stopwords normales
stop_words = set(stopwords.words('spanish'))


# Stopwords personalizadas
extra_stopwords = {
    "proyecto",
    "fortalecimiento",
    "desarrollo",
    "procesos",
    "gestión",
    "actividades",
    "mediante",
    "personas",
    "sector",
    "comunidades",
    "población",
    "de",
    "del",
    "en",
    "el",
    "la"
}

# Unir ambas
all_stopwords = stop_words.union(extra_stopwords)

In [41]:
for columna in df_final.columns[18:]:

      positivos = df_final[
          df_final[columna] == 1
      ]

      texto_total = " ".join(
          positivos["texto"].astype(str)
      )

      # Minúsculas
      texto_total = texto_total.lower()

      # Quitar puntuación
      texto_total = re.sub(
          r'[^\w\s]',
          '',
          texto_total
      )

      # Separar palabras
      palabras = texto_total.split()

      # Filtrar stopwords
      palabras_filtradas = [
          palabra
          for palabra in palabras
          if palabra not in all_stopwords
      ]

      # Contar
      conteo = Counter(
          palabras_filtradas
      )

      print("\n================")
      print(columna)
      print("================")

      print(
          conteo.most_common(20)
      )


A.1.2 Suelo
[('comunes', 14), ('bienes', 13), ('saberes', 8), ('tierra', 7), ('prácticas', 7), ('socioambientales', 7), ('proyectos', 6), ('programa', 6), ('observatorio', 6), ('espacios', 5), ('través', 5), ('información', 5), ('cuido', 5), ('defensa', 5), ('desigualdades', 5), ('bailes', 4), ('folklóricos', 4), ('social', 4), ('integral', 4), ('grupo', 4)]

A.1.3 Ambiente, Descarbonización y Residuos
[('ambiental', 20), ('residuos', 14), ('sostenible', 13), ('salud', 11), ('trabajo', 10), ('región', 10), ('ambiente', 10), ('caribe', 10), ('conservación', 10), ('laboratorio', 9), ('manejo', 9), ('materiales', 8), ('educación', 8), ('limón', 8), ('instituciones', 7), ('comunal', 7), ('provincia', 7), ('servicios', 6), ('área', 6), ('1', 6)]

A.2.3 Cultura
[('cultural', 256), ('patrimonio', 232), ('artes', 199), ('cultura', 174), ('estudiantes', 139), ('social', 136), ('comunidad', 121), ('espacios', 115), ('través', 115), ('educativo', 103), ('costa', 99), ('1', 97), ('medio', 95), ('

In [42]:
conteos = {}

for columna in df_final.columns[18:]:

    cantidad = (
        df_final[columna] == 1
    ).sum()

    conteos[columna] = cantidad

conteos_ordenados = sorted(
    conteos.items(),
    key=lambda x: x[1],
    reverse=True
)

for nombre, cantidad in conteos_ordenados:
  if (nombre != "texto"):
    print(nombre, ":", cantidad)

C.3.1 Educación : 176
H.5.2 Mejora rendimiento académico estudiantes de cantones de bajo IDS : 137
B.2.2 Embarazos adolescentes : 123
C.4.2 Salud : 118
F.1.2 Educación continua y permanente : 115
F.7.1 Defensa ambiente sustentable : 102
A.2.3 Cultura : 99
D.2.1 Productividad laboral : 97
A.3.1 Ciencia, Tecnología e Innovación : 53
F.7.3 Contribuye a la seguridad alimentaria : 30
C.4.1 Género : 22
B.1.2 Alimentación sostenible en zonas costeras : 19
E.2.3 Inseguridad Ciudadana : 13
B.4.1 Trabajo digno : 12
C.5.2 Producción sofisticada : 9
A.1.1 Agua : 8
A.1.3 Ambiente, Descarbonización y Residuos : 5
F.7.2 Descarbonización : 5
H.5.1 Atención cantones del país debajo índice de desarrollo social (IDS) : 5
C.3.2 Innovación : 4
D.1.1 Descentralizada : 4
D.1.2 Digitalizada : 4
A.3.3 Vivienda y Hábitat : 3
B.5.4 Ambiente y Mares : 3
C.1.1 Desarrollo humano integral : 3
F.1.1 Bienestar nacional global : 3
H.6.3 Fortalecimiento interuniversitario de emprendimientos regionales : 3
A.1.2 Suelo : 

## Intento de predicciones

In [43]:
X = df_final["texto"]

y = df_final["A.2.3 Cultura"]

#y = df_final["C.3.1 Educación"]

#y = df_final["A.1.2 Suelo"]

#y = df_final["H.5.2 Mejora rendimiento académico estudiantes de cantones de bajo IDS"]

In [44]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [45]:
vectorizer = TfidfVectorizer(stop_words=list(all_stopwords))

X_train_tfidf = vectorizer.fit_transform(X_train)

X_test_tfidf = vectorizer.transform(X_test)

In [46]:
modelo = LogisticRegression(
    max_iter=1000,
    class_weight='balanced'
)

In [47]:
modelo.fit(X_train_tfidf, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

In [48]:
predicciones = modelo.predict(X_test_tfidf)

In [49]:
accuracy = accuracy_score(
    y_test,
    predicciones
)

print("Accuracy:", accuracy)

Accuracy: 0.9655172413793104


In [50]:
print(
    classification_report(
        y_test,
        predicciones
    )
)

              precision    recall  f1-score   support

           0       0.96      1.00      0.98        98
           1       1.00      0.78      0.88        18

    accuracy                           0.97       116
   macro avg       0.98      0.89      0.93       116
weighted avg       0.97      0.97      0.96       116



In [51]:
feature_names = vectorizer.get_feature_names_out()

coeficientes = modelo.coef_[0]

top_positivas = sorted(
    zip(coeficientes, feature_names),
    reverse=True
)[:20]

print(top_positivas)

[(np.float64(3.535026764347605), 'artes'), (np.float64(2.609490093583732), 'cultural'), (np.float64(2.1813564767727183), 'patrimonio'), (np.float64(2.0756514678805535), 'cultura'), (np.float64(1.5980827023755353), 'arte'), (np.float64(1.5803027633792495), 'tradiciones'), (np.float64(1.4111147297347164), 'musical'), (np.float64(1.3302564367745398), 'música'), (np.float64(1.2644416503241733), 'musicales'), (np.float64(1.223933477728867), 'cine'), (np.float64(0.9949879722728471), 'teatro'), (np.float64(0.9398054942908749), 'artística'), (np.float64(0.9188464124101772), 'agua'), (np.float64(0.787401312309558), 'indígenas'), (np.float64(0.7550448081626076), 'danza'), (np.float64(0.7310926357131795), 'artísticas'), (np.float64(0.7307993404044771), 'museo'), (np.float64(0.7201980008982318), 'colecciones'), (np.float64(0.7140966951921868), 'turrialba'), (np.float64(0.7020389984245733), 'diseño')]
